# Step 1: Install Dependencies

This notebook uses Unsloth models exclusively.

In [3]:
# Install required packages for this notebook.
%pip install -q unsloth datasets trl accelerate transformers wandb optuna       # mlx-tune on MacOS

# Step 1a: Login to wandb

Execute once to login to your wandb account. You can create a free account at https://wandb.ai/site.

In [1]:
import wandb
import os
os.environ["WANDB_API_KEY"] = "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"

# Step 2: Load Model

We load the model using unsloth, which works excellent on CUDA.

In [2]:
from unsloth import FastLanguageModel
import torch

# Model configuration
max_seq_length = 2048  # Supports up to RoPE Scaling internally  
dtype = None  # Auto-detect: Float16 for older GPUs, Bfloat16 for Ampere+
load_in_4bit = True  # Use 4-bit quantization to reduce memory

# Load pre-trained base model and tokenizer
base, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # Use if accessing gated models
)

# Verify base model loaded
print(f"✓ Base model loaded: {base.config._name_or_path}")
print(f"✓ Max sequence length: {max_seq_length}")
print(f"✓ Device: {next(base.parameters()).device}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.6.2. vLLM: 0.19.1.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Unsloth: Will load unsloth/llama-3.2-3b-unsloth-bnb-4bit as a legacy tokenizer.


✓ Base model loaded: unsloth/llama-3.2-3b-unsloth-bnb-4bit
✓ Max sequence length: 2048
✓ Device: cuda:0


# Step 3: Attach LoRA Adapters

Parameters: rank=16, alpha=16.


In [3]:
# Configure and attach LoRA adapters
model = FastLanguageModel.get_peft_model(
    base,
    r = 16,  # LoRA rank - higher = more parameters, better quality
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,  # LoRA scaling - typically equal to r
    lora_dropout = 0,  # Supports any, but = 0 is optimized
    bias = "none",  # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth",  # 30% longer; saves VRAM
    random_state = 3407,
    use_rslora = False,  # Rank stabilized LoRA
    loftq_config = None,  # LoftQ quantization
)

print("✓ LoRA adapters attached")
print(f"✓ Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


[transformers] Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✓ LoRA adapters attached
✓ Trainable parameters: 24,313,856


# Step 4: Load Dataset (from HF to local cache)

This cell loads a dataset from Hugging Face and saves it to the local cache directory.

In [4]:
from datasets import load_dataset

HF_DATASET_ID = "ServiceNow-AI/R1-Distill-SFT"

# Load dataset from Hugging Face
dataset = load_dataset(HF_DATASET_ID, 'v0', split="train")

# Use subset for faster experimentation (remove [:5000] for full dataset)
dataset = dataset.select(range(5000))

print(f"✓ Dataset loaded: {len(dataset):,} examples")
print(f"✓ Columns: {dataset.column_names}")
print(f"\\nSample:")
print(dataset[0]['problem'][:200] + "...")

✓ Dataset loaded: 5,000 examples
✓ Columns: ['id', 'reannotated_assistant_content', 'problem', 'source', 'solution', 'verified', 'quality_metrics']
\nSample:
There were 27 boys and 35 girls on the playground at recess. There were _____ children on the playground at recess....


# Step 5: Convert Records to Chat Training Text

SFT expects text in the model's chat format. We transform each example into:
- one user message (`problem`)
- one assistant message (`reannotated_assistant_content`)

Then we apply the tokenizer chat template so training examples match inference-time prompting style.

In [5]:
from unsloth.chat_templates import get_chat_template

# Apply Llama 3 chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",  # Llama 3.2 uses 3.1 template
)

# Format function for the dataset
def formatting_prompts_func(examples):
    convos = []
    for problem, response in zip(examples["problem"], examples["reannotated_assistant_content"]):
        convo = [
            {"role": "user", "content": problem},
            {"role": "assistant", "content": response},
        ]
        convos.append(convo)
    
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
             for convo in convos]
    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

print("✓ Dataset formatted with chat template")
print(f"\\nFormatted example:\\n{dataset[0]['text'][:300]}...")


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

✓ Dataset formatted with chat template
\nFormatted example:\n<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

There were 27 boys and 35 girls on the playground at recess. There were _____ children on the playground at recess.<|eot_id...


# Step 6: Configure Trainer

We use TRL's SFTTrainer with TrainingArguments. This works on both CUDA and MPS.

AdamW optimizer is used uniformly.


In [6]:
# Configure Trainer with TRL

MAX_STEPS_QUICK_RUN = -1             # set to +ve number for quick testing
PER_DEVICE_TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1     # Effective batch size = 16
WARMUP_STEPS = 10
NUM_TRAIN_EPOCHS = 3                # Set to 1 for quick test
LEARNING_RATE = 2e-4

# Use AdamW optimizer
optimizer_name = "adamw_8bit"       # Use 8-bit AdamW if available for faster training and lower memory
    
# optimizer_name = "adamw_torch"

from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments, DataCollatorForSeq2Seq, DataCollatorForLanguageModeling

# Dataset already has input_ids + labels from Step 5; no formatting_func needed
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    packing = False,  # Can make training 5x faster for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
        warmup_steps = WARMUP_STEPS,
        num_train_epochs = NUM_TRAIN_EPOCHS,
        learning_rate = LEARNING_RATE,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = optimizer_name,
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",  # Use wandb for experiment tracking
        run_name="Llama-3.2-3B-finetune-trl-local"
    ),
)

print("✓ Trainer configured")
print(f"✓ Optimizer: {optimizer_name}")
print(f"✓ Max steps: {MAX_STEPS_QUICK_RUN}")


Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/5000 [00:00<?, ? examples/s]

✓ Trainer configured
✓ Optimizer: adamw_8bit
✓ Max steps: -1


# Step 7: Train the Model

Before training, we print device-aware memory information when available.

In [ ]:
# Show GPU memory stats before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU: {gpu_stats.name}")
print(f"Max memory: {max_memory} GB")
print(f"Reserved: {start_gpu_memory} GB\\n")

# Train the model
trainer_stats = trainer.train()

# Show final memory stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_training = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)

print(f"\\n✓ Training complete!")
print(f"Peak memory: {used_memory} GB ({used_percentage}% of {max_memory} GB)")
print(f"Training memory: {used_memory_training} GB")
print(f"Final loss: {trainer_stats.training_loss:.4f}")

GPU: NVIDIA A100 80GB PCIe
Max memory: 79.251 GB
Reserved: 2.359 GB\n


[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 3 | Total steps = 1,875
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: tango16 (tango16-whisky) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.189604
20,0.911738
30,0.808604
40,0.774372
50,0.723631
60,0.697856
70,0.684207
80,0.720146
90,0.674479
100,0.682390


[transformers] Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1500/tokenizer_config.json.
[transformers] Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1875/tokenizer_config.json.


\n✓ Training complete!
Peak memory: 24.186 GB (30.518% of 79.251 GB)
Training memory: 21.827 GB
Final loss: 0.5429


# Step 8: Quick Inference Sanity Check

After training, switch to inference mode and test a reasoning prompt.

We move inputs to the same detected device used during setup, so this cell works across CUDA, MPS, and CPU.

In [41]:
# Enable fast inference mode
FastLanguageModel.for_inference(model)

# Test with a logic puzzle
test_problem = """If Alex is taller than Blake, Blake is taller than Casey, 
and Casey is taller than Dana, who is the shortest person?"""

# Format as chat
messages = [{"role": "user", "content": test_problem}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# Generate response
from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer, skip_prompt = True)
outputs = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.7,
    top_p = 0.9,
)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>
First, I need to understand the relationships between the people's heights. The problem states that Alex is taller than Blake, Blake is taller than Casey, and Casey is taller than Dana.

To find the shortest person, I'll start by analyzing the relationships between the people's heights:

1. Alex is taller than Blake.
2. Blake is taller than Casey.
3. Casey is taller than Dana.

From these relationships, I can infer the following:

- Alex is taller than Blake.
- Blake is taller than Casey.
- Casey is taller than Dana.

This suggests that Alex is taller than Dana. To confirm this, I'll consider the height of each person:

- Alex: taller than Blake.
- Blake: taller than Casey.
- Casey: taller than Dana.

This shows that Alex is taller than Dana. Therefore, Dana is the shortest person.
</think>

To determine who is the shortest person, let's analyze the relationships between the people's heights step by step.

1. **Alex is taller than Blake:**
   - Alex > Blake

2. **Blake is talle

# Step 9: Save Artifacts

You can save:
- LoRA adapters only (small and convenient),
- merged full model weights,
- GGUF for local runtimes.

Keep Hub upload optional unless your token and repo are configured.

In [ ]:
# Save LoRA adapters (works for both Unsloth and Transformers+PEFT)
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# Option 2: Save merged model (larger ~6GB for 3B)
model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit")

# # Option 3: Save as GGUF for Ollama
# model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")

print("✓ Model artifacts saved")

Found HuggingFace hub cache directory: /home/tango/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 8456.26it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:28<00:00, 14.37s/it]


Unsloth: Merge process complete. Saved to `/home/tango/model`
✓ Model artifacts saved


# Step 10: Hyperparameter Auto-Tuning with Optuna

This section runs automatic hyperparameter search over:
- `learning_rate` (log scale)
- LoRA `rank`
- `lora_alpha`

For each trial, we reload a fresh base model, attach LoRA adapters, do a short training run, and return final loss.

Default trial count is set to 20 (you can reduce it for faster experimentation).

In [18]:
import optuna

NO_OF_TRIALS = 10

def objective(trial):
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    r = trial.suggest_categorical("rank", [8, 16, 32])
    alpha = trial.suggest_categorical("lora_alpha", [8, 16, 32])

    trial_model = FastLanguageModel.get_peft_model(
        base,
        r=r,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=alpha,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )

    trial_trainer = SFTTrainer(
        model=trial_model,
        processing_class=tokenizer,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            warmup_steps=WARMUP_STEPS,
            max_steps=50,          # short run per trial
            learning_rate=lr,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir=f"outputs/trial_{trial.number}",
            report_to="none",
        ),
    )

    result = trial_trainer.train()
    trial_model.unload()  # Free up VRAM after each trial
    return result.training_loss

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=NO_OF_TRIALS)

best = study.best_params
print(f"Best params: {best}")


[I 2026-04-27 12:57:20,425] A new study created in memory with name: no-name-edbd340f-9988-46b7-80e2-d85181cf79dd
/home/tango/miniconda3/envs/msds/lib/python3.11/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/tango/miniconda3/envs/msds/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters =

Step,Training Loss
10,0.948769
20,0.788292
30,0.729027
40,0.685861
50,0.621630


[I 2026-04-27 13:00:49,475] Trial 0 finished with value: 0.754715871810913 and parameters: {'learning_rate': 5.5928311302985984e-05, 'rank': 8, 'lora_alpha': 25}. Best is trial 0 with value: 0.754715871810913.
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 48,627,712 of 3,261,377,536 (1.49% trained)


Step,Training Loss
10,0.966254
20,0.835100
30,0.776403
40,0.733947
50,0.659605


[transformers] Unsloth: Restored added_tokens_decoder metadata in outputs/trial_1/checkpoint-50/tokenizer_config.json.
[I 2026-04-27 13:04:23,916] Trial 1 finished with value: 0.794262056350708 and parameters: {'learning_rate': 7.069575696281574e-05, 'rank': 32, 'lora_alpha': 12}. Best is trial 0 with value: 0.754715871810913.
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 48,627,712 of 3,261,377,536 (1.49% trained)


...

Step,Training Loss
10,0.841356
20,0.583303
30,0.574246
40,0.555769
50,0.511701


[transformers] Unsloth: Restored added_tokens_decoder metadata in outputs/trial_8/checkpoint-50/tokenizer_config.json.
[I 2026-04-27 13:29:37,273] Trial 8 finished with value: 0.6132750129699707 and parameters: {'learning_rate': 0.0002020752984232237, 'rank': 32, 'lora_alpha': 44}. Best is trial 6 with value: 0.5877984523773193.
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 12,156,928 of 3,224,906,752 (0.38% trained)


Step,Training Loss
10,0.919549
20,0.715928
30,0.659785
40,0.630348
50,0.579270


[transformers] Unsloth: Restored added_tokens_decoder metadata in outputs/trial_9/checkpoint-50/tokenizer_config.json.
[I 2026-04-27 13:33:12,639] Trial 9 finished with value: 0.7009759712219238 and parameters: {'learning_rate': 4.60789598299741e-05, 'rank': 8, 'lora_alpha': 60}. Best is trial 6 with value: 0.5877984523773193.


Best params: {'learning_rate': 0.00028321697940604686, 'rank': 16, 'lora_alpha': 54}
